In [ ]:
# # 03_eda.ipynb - Análisis Exploratorio de Datos
# Data Analyst
# ============================================
# 1. IMPORTS
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías cargadas")

In [ ]:
# ============================================
# 2. CARGA DE DATOS (asume que el archivo existe)
# ============================================
from pathlib import Path

DATA_PATH = Path("../data/raw/dataset_pucp.csv")

# Verificación amigable (no bloqueante)
if not DATA_PATH.exists():
    print(f"❌ El archivo no está en: {DATA_PATH}")
    print("📌 Por favor, asegúrate de que el dataset esté en data/raw/dataset_pucp.csv")
else:
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
    
# Si no está, puedes crear un mensaje claro pero NO detener el flujo

In [ ]:
# ============================================
# 3. VERIFICACIÓN INICIAL
# ============================================
if 'df' in locals():
    print("\n📊 PRIMERAS 5 FILAS:")
    display(df.head())
    
    print("\n📊 TIPOS DE DATOS:")
    print(df.dtypes.value_counts())
    
    print("\n📊 ESTADÍSTICAS BÁSICAS:")
    display(df.describe())
else:
    print("⚠️ No se pudieron cargar los datos. Verifica la ruta.")

In [ ]:
# ============================================
# 4. CALIDAD DE DATOS
# ============================================
if 'df' in locals():
    print("="*60)
    print("📋 ANÁLISIS DE CALIDAD")
    print("="*60)
    
    # Nulos
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df)) * 100
    nulos_df = pd.DataFrame({'Nulos': nulos, '%': nulos_pct})
    print("\n📌 VALORES NULOS:")
    print(nulos_df[nulos_df['Nulos'] > 0] if (nulos > 0).any() else "✅ No hay valores nulos")
    
    # Duplicados
    dups = df.duplicated().sum()
    print(f"\n📌 FILAS DUPLICADAS: {dups} ({dups/len(df)*100:.2f}%)")
    
    # Tipos
    print("\n📌 TIPOS DE DATOS:")
    print(df.dtypes.value_counts())

In [ ]:
# ============================================
# 5. ANÁLISIS DESCRIPTIVO
# ============================================
if 'df' in locals():
    print("="*60)
    print("📊 ESTADÍSTICAS DESCRIPTIVAS")
    print("="*60)
    
    # Numéricas
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    print("\n📌 VARIABLES NUMÉRICAS:")
    display(df[numeric_cols].describe())
    
    # Categóricas
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    if len(categorical_cols) > 0:
        print("\n📌 VARIABLES CATEGÓRICAS (top valores):")
        for col in categorical_cols[:3]:
            print(f"\n--- {col} ---")
            print(df[col].value_counts().head())

In [ ]:
# ============================================
# 6. BALANCE DE CLASES (TARGET)
# ============================================
if 'df' in locals():
    print("="*60)
    print("🎯 BALANCE DEL TARGET")
    print("="*60)
    
    # Identificar target (probablemente 'Revenue')
    target = 'Revenue' if 'Revenue' in df.columns else df.columns[-1]
    print(f"Target identificado: {target}")
    
    counts = df[target].value_counts()
    pct = df[target].value_counts(normalize=True) * 100
    
    print(f"\nDistribución:")
    for val, count, perc in zip(counts.index, counts.values, pct.values):
        print(f"  {val}: {count} ({perc:.1f}%)")
    
    # Visualización
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(counts.index.astype(str), counts.values, color=['#FF6B6B', '#4ECDC4'])
    axes[0].set_title('Distribución de clases')
    axes[1].pie(counts.values, labels=counts.index.astype(str), autopct='%1.1f%%', 
                colors=['#FF6B6B', '#4ECDC4'])
    axes[1].set_title('Porcentaje')
    plt.tight_layout()
    plt.show()
    
    # Conclusión
    min_class = pct.min()
    if min_class < 30:
        print(f"\n⚠️ Dataset DESBALANCEADO (clase minoritaria: {min_class:.1f}%)")
    else:
        print(f"\n✅ Dataset balanceado")

In [ ]:
# ============================================
# 7. OUTLIERS
# ============================================
if 'df' in locals():
    print("="*60)
    print("📌 DETECCIÓN DE OUTLIERS (IQR)")
    print("="*60)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    outliers_summary = []
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        pct_outliers = (n_outliers / len(df)) * 100
        outliers_summary.append({'Variable': col, 'Outliers': n_outliers, '%': pct_outliers})
    
    outliers_df = pd.DataFrame(outliers_summary).sort_values('%', ascending=False)
    print(outliers_df[outliers_df['Outliers'] > 0].head(10))
    
    # Boxplots
    n_plots = min(6, len(numeric_cols))
    fig, axes = plt.subplots(1, n_plots, figsize=(18, 4))
    for i, col in enumerate(numeric_cols[:n_plots]):
        axes[i].boxplot(df[col].dropna())
        axes[i].set_title(col)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================
# 8. MATRIZ DE CORRELACIÓN
# ============================================
if 'df' in locals():
    print("="*60)
    print("🔗 CORRELACIONES")
    print("="*60)
    
    numeric_df = df.select_dtypes(include=[np.number])
    corr_matrix = numeric_df.corr()
    
    # Heatmap
    plt.figure(figsize=(14, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=0.5)
    plt.title('Matriz de correlación', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Correlaciones con target
    if target in corr_matrix.columns:
        print(f"\n📌 Top correlaciones con '{target}':")
        corr_target = corr_matrix[target].sort_values(ascending=False)
        print(corr_target)
        
        # Visualizar top 5
        top5 = corr_target[1:6]
        plt.figure(figsize=(10, 5))
        top5.plot(kind='barh', color='teal')
        plt.title(f'Top 5 variables correlacionadas con {target}')
        plt.xlabel('Correlación')
        plt.tight_layout()
        plt.show()

In [ ]:
# ============================================
# 9. HISTOGRAMAS Y DISTRIBUCIONES
# ============================================
if 'df' in locals():
    print("="*60)
    print("📈 DISTRIBUCIONES")
    print("="*60)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    n_cols = min(9, len(numeric_cols))
    rows = (n_cols + 2) // 3
    
    fig, axes = plt.subplots(rows, 3, figsize=(15, rows*4))
    axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols[:n_cols]):
        axes[i].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
        axes[i].set_title(f'{col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frecuencia')
    
    for i in range(n_cols, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================
# 10. CONCLUSIONES
# ============================================
if 'df' in locals():
    print("="*60)
    print("📝 RESPUESTAS A PREGUNTAS GUÍA")
    print("="*60)
    
    print(f"1. ¿Registros y variables? {df.shape[0]} filas, {df.shape[1]} columnas")
    
    nulos_total = df.isnull().sum().sum()
    print(f"2. ¿Porcentaje de nulos? {nulos_total/len(df)/df.shape[1]*100:.2f}% del total")
    
    if target in df.columns:
        min_class = df[target].value_counts(normalize=True).min()
        print(f"3. ¿Balance de clases? {'Desbalanceado' if min_class < 0.3 else 'Balanceado'} ({min_class:.1f}% clase menor)")
    
    print(f"4. ¿Variables numéricas? {len(df.select_dtypes(include=[np.number]).columns)}")
    print(f"5. ¿Variables categóricas? {len(df.select_dtypes(include=['object']).columns)}")
    
    print("\n" + "="*60)
    print("✅ EDA COMPLETADO")
    print("="*60)